In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk
#from e_1_run_cvae_gpu import train_chunk
#from e_1_run_cvae_time_check import train_chunk
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# training

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [512, 256, 128] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 94
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.05
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt"

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=0->94 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=94 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -2.2136 | KL: 2.0874 | Total: -0.1262
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -3.9883 | KL: 3.4529 | Total: -0.5354
Chunk step     3 | epoch    1 chunk   3/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -4.3717 | KL: 3.7823 | Total: -0.5894
Chunk step     4 | epoch    1 chunk   4/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -4.4648 | KL: 3.8502 | Total: -0.6146
Chunk step     5 | epoch    1 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -4.5098 | KL: 3.8786 | Total: -0.6311
Chunk step     6 | epoch    1 chunk   6/97 | file_idx  66 | BN off 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2034.pt | 완료 chunks=2034
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=2034->2037 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2035 | epoch   21 chunk  95/97 | file_idx  97 | BN off    | beta_eff: 1.0000 | Recon: -5.8221 | KL: 5.0684 | Total: -0.7537
Validation @ chunk  2035 | Recon: -5.8222 | KL: 5.0768 | Total: -0.7454 | KL_dim: [1.492518, 3.584224]
Chunk step  2036 | epoch   21 chunk  96/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.8181 | KL: 5.0850 | Total: -0.7331
Validation @ chunk  2036 | Recon: -5.8167 | KL: 5.0722 | Total: -0.7445 | KL_dim: [1.49189, 3.580346]
Chunk step  2037 | epoch   21 chunk  97/97 | file_idx   2 | BN off    | beta_eff: 1.0000 | Recon: -5.8217 | KL: 5.0736 | Total: -0.7480
Validation @ chunk  2

In [ ]:
num_chunks  = 94
val_every_chunks = 94
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2037.pt | 완료 chunks=2037
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=2037->2131 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=94 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2038 | epoch   22 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -5.8226 | KL: 5.0675 | Total: -0.7551
Chunk step  2039 | epoch   22 chunk   2/97 | file_idx  86 | BN off    | beta_eff: 1.0000 | Recon: -5.8207 | KL: 5.0759 | Total: -0.7448
Chunk step  2040 | epoch   22 chunk   3/97 | file_idx  29 | BN off    | beta_eff: 1.0000 | Recon: -5.8228 | KL: 5.0721 | Total: -0.7506
Chunk step  2041 | epoch   22 chunk   4/97 | file_idx  20 | BN off    | beta_eff: 1.0000 | Recon: -5.8278 | KL: 5.0573 | Total: -0.7705
Chunk step  2042 | epoch   22 chunk   5/97 | file_idx  70 | BN off    | beta_eff: 1.0000

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2131.pt | 완료 chunks=2131
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=2131->2134 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2132 | epoch   22 chunk  95/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | Recon: -5.8241 | KL: 5.0707 | Total: -0.7534
Validation @ chunk  2132 | Recon: -5.8224 | KL: 5.0747 | Total: -0.7478 | KL_dim: [1.491617, 3.583022]
Chunk step  2133 | epoch   22 chunk  96/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.8221 | KL: 5.0780 | Total: -0.7441
Validation @ chunk  2133 | Recon: -5.8259 | KL: 5.0780 | Total: -0.7479 | KL_dim: [1.491975, 3.585984]
Chunk step  2134 | epoch   22 chunk  97/97 | file_idx  87 | BN off    | beta_eff: 1.0000 | Recon: -5.8191 | KL: 5.0833 | Total: -0.7357
Validation @ chunk  

In [ ]:
num_chunks  = 94
val_every_chunks = 94
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2134.pt | 완료 chunks=2134
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=2134->2228 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=94 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2135 | epoch   23 chunk   1/97 | file_idx  19 | BN off    | beta_eff: 1.0000 | Recon: -5.8245 | KL: 5.0741 | Total: -0.7504
Chunk step  2136 | epoch   23 chunk   2/97 | file_idx  86 | BN off    | beta_eff: 1.0000 | Recon: -5.8220 | KL: 5.0771 | Total: -0.7449
Chunk step  2137 | epoch   23 chunk   3/97 | file_idx   0 | BN off    | beta_eff: 1.0000 | Recon: -5.8212 | KL: 5.0771 | Total: -0.7442
Chunk step  2138 | epoch   23 chunk   4/97 | file_idx  53 | BN off    | beta_eff: 1.0000 | Recon: -5.8216 | KL: 5.0773 | Total: -0.7443
Chunk step  2139 | epoch   23 chunk   5/97 | file_idx  22 | BN off    | beta_eff: 1.0000

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2228.pt | 완료 chunks=2228
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=2228->2231 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2229 | epoch   23 chunk  95/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.8248 | KL: 5.0785 | Total: -0.7462
Validation @ chunk  2229 | Recon: -5.8243 | KL: 5.0774 | Total: -0.7469 | KL_dim: [1.490039, 3.587397]
Chunk step  2230 | epoch   23 chunk  96/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.8284 | KL: 5.0728 | Total: -0.7556
Validation @ chunk  2230 | Recon: -5.8256 | KL: 5.0793 | Total: -0.7463 | KL_dim: [1.490857, 3.588425]
Chunk step  2231 | epoch   23 chunk  97/97 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -5.8229 | KL: 5.0792 | Total: -0.7438
Validation @ chunk  

In [9]:
num_chunks  = 94
val_every_chunks = 94
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2231.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2325.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2231.pt | 완료 chunks=2231
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=2231->2325 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=94 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2232 | epoch   24 chunk   1/97 | file_idx  39 | BN off    | beta_eff: 1.0000 | Recon: -5.8250 | KL: 5.0767 | Total: -0.7483
Chunk step  2233 | epoch   24 chunk   2/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.8235 | KL: 5.0728 | Total: -0.7507
Chunk step  2234 | epoch   24 chunk   3/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.8256 | KL: 5.0661 | Total: -0.7595
Chunk step  2235 | epoch   24 chunk   4/97 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -5.8233 | KL: 5.0792 | Total: -0.7441
Chunk step  2236 | epoch   24 chunk   5/97 | file_idx   6 | BN off    | beta_eff: 1.0000

In [10]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2325.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2328.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk2325.pt | 완료 chunks=2325
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=2325->2328 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  2326 | epoch   24 chunk  95/97 | file_idx  90 | BN off    | beta_eff: 1.0000 | Recon: -5.8298 | KL: 5.0757 | Total: -0.7541
Validation @ chunk  2326 | Recon: -5.8326 | KL: 5.0851 | Total: -0.7475 | KL_dim: [1.491532, 3.593549]
Chunk step  2327 | epoch   24 chunk  96/97 | file_idx  30 | BN off    | beta_eff: 1.0000 | Recon: -5.8288 | KL: 5.0837 | Total: -0.7451
Validation @ chunk  2327 | Recon: -5.8248 | KL: 5.0785 | Total: -0.7463 | KL_dim: [1.489305, 3.589242]
Chunk step  2328 | epoch   24 chunk  97/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -5.8254 | KL: 5.0898 | Total: -0.7356
Validation @ chunk  

# BN = 5

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN frozen | beta_eff: 1.0000 | Recon: -5.3391 | KL: 4.5933 | Total: -0.7458
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN frozen | beta_eff: 1.0000 | Recon: -5.4229 | KL: 4.6885 | Total: -0.7345
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN frozen | beta_eff: 1.0000 | Recon: -5.4530 | KL: 4.7225 | Total: -0.7305
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN frozen | beta_eff: 1.0000 | Recon: -5.4805 | KL: 4.7367 | Total: -0.7437
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN frozen | beta_eff: 1.0000 | Recon: -5.4988 | KL: 4.7528 | Total: -0.7460
Chunk ste

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_3-0.0001_4-1e-06_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=5 | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN frozen | beta_eff: 1.0000 | Recon: -5.2304 | KL: 4.4934 | Total: -0.7370
Validation @ chunk   384 | Recon: -5.2282 | KL: 4.4857 | Total: -0.7425 | KL_dim: [1.436707, 3.048991]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN frozen | beta_eff: 1.0000 | Recon: -5.2268 | KL: 4.5010 | Total: -0.7258
Validation @ chunk   385 | Recon: -5.2273 | KL: 4.4858 | Total: -0.7415 | KL_dim: [1.42515, 3.060651]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN frozen | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.4922 | Total: -0.7463
Validation @ chunk   386 | Recon: -5.2095 | KL: 4.4750 | Total: -0.7345 | KL_dim: [1.42636

# X,M norm

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001,0.0003, 0.0005
beta        = 0.9
warmup_chunks = None # None or num
num_chunks  = 70
validation_chunk_idxs = [15,24,78]
val_every_chunks = 3
resume_path = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk30.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk100.pt"

In [6]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_norm(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    x_mean=bs_stats["x_mean"],
    x_std=bs_stats["x_std"],
    m_mean=bs_stats["m_mean"],
    m_std=bs_stats["m_std"],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae_xm_norm/bs/cvae_bs_8_128_4096_None_0.001_0.9_None_chunk30.pt | 완료 chunks=30
학습 시작 | 이번 실행 chunks=70 | 진행 chunks=30->100 | files/epoch=100 | bn_chunks=None | warmup_chunks=None
Chunk step    31 | epoch    1 chunk  31/100 | file_idx  30 | BN off    | beta_eff: 0.9000 | Recon: -4.9477 | KL: 5.3827 | Total: -0.1033
Chunk step    32 | epoch    1 chunk  32/100 | file_idx  29 | BN off    | beta_eff: 0.9000 | Recon: -4.8912 | KL: 6.1971 | Total: 0.6862
Chunk step    33 | epoch    1 chunk  33/100 | file_idx  79 | BN off    | beta_eff: 0.9000 | Recon: -4.9538 | KL: 6.2571 | Total: 0.6776
Chunk step    34 | epoch    1 chunk  34/100 | file_idx  44 | BN off    | beta_eff: 0.9000 | Recon: -5.0059 | KL: 5.4327 | Total: -0.1164
Chunk step    35 | epoch    1 chunk  35/100 | file_idx  71 | BN off    | beta_eff: 0.9000 | Recon: -4.9941 | KL: 5.4282 | Total: -0.1088
Chunk step    36 | epoch    1 chunk  36/100 | file_idx  66 | BN off    | beta_eff: 0.9000 | Rec